# pdf reader and analyser

In [1]:
import os
from typing import TypedDict
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

In [3]:
class strstage(TypedDict):
    question: str
    context : str
    draft: str
    answer: str

def pdf_reader():
    path = input("Enter the name and path of pdf file:")
    print("the file is loading...")
    
    loader= PyPDFLoader(path)
    doc = loader.load()
    txt_spliter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap= 100)
    chunks = txt_spliter.split_documents(doc)
    
    embed = OllamaEmbeddings(model="nomic-embed-text")
    vector_store = Chroma.from_documents(chunks, embed)
    retr = vector_store.as_retriever(search_kwargs={"k":5})
    
    llm = ChatOllama(model="qwen3:4b", temperature = 0)
    
    
    def retrieve(state: strstage):
        docs = retr.invoke(state["question"])
        context_str = "\n\n".join(d.page_content for d in docs)
        return{"context": context_str}
    
    def draft(state: strstage):
        prompt = ChatPromptTemplate.from_template(
            "answer the question: {question}\n based only on thid context: {context}"
        )
        chain = prompt | llm | StrOutputParser()
        draft = chain.invoke({"context": state["context"], "question": state["question"]})
        return {"draft": draft}
    
    def final(state: strstage):
        refine_prompt = ChatPromptTemplate.from_template(
            """you are a expert editor
            review the given draft answer in regard to the given contect and it must perfectly answer the User's question 
            without any hallucination, with all errors fixed, and make it professional and consise. information must only 
            be from the given context strictly.
            
            context: {context}
            User's question: {question}
            draft: {draft}
            
            final Answer:
            """
        )
        chain = refine_prompt | llm | StrOutputParser()
        final = chain.invoke({
            "context": state["context"],
            "question": state["question"],
            "draft" : state["draft"]
        })
        return {"answer": final}
    flow = StateGraph(strstage)
    flow.add_node("retrieve", retrieve)
    flow.add_node("draft", draft)
    flow.add_node("final", final)
    
    flow.set_entry_point("retrieve")
    flow.add_edge("retrieve", "draft")
    flow.add_edge("draft", "final")
    flow.add_edge("final", END)
    
    model = flow.compile()
    print("Loading Done!!")
    
    while True:
        que = input("Ask from the pdf you gave:")
        if que.lower() in ['quit', 'exit']:
            print("Bye!")
            break
        print("question processing")
        
        ans = model.invoke({"question": que})
        
        print("Final answer: ")
        print(ans["answer"])
    
    
pdf_reader()

Enter the name and path of pdf file:RLM.pdf
the file is loading...
Loading Done!!
Ask from the pdf you gave:exit
Bye!


In [4]:
!pip freeze > requirements.txt